# AI Engineer Roadmap Code Lab — Colab Edition

**Goal:** become a practical AI Engineer who can build, evaluate, and ship AI-powered products.

Based on:
- roadmap.sh AI Engineer roadmap: https://roadmap.sh/ai-engineer
- roadmap.sh AI Engineer projects: https://roadmap.sh/ai-engineer/projects
- Useful GitHub curricula discovered during research:
  - https://github.com/dswh/ai-engineer-roadmap
  - https://github.com/PavanMudigonda/zero-to-ai
  - https://github.com/AgenticAiLabs/Ai-Engineering-Roadmap
  - https://github.com/alexeygrigorev/ai-engineering-field-guide
  - https://github.com/mlabonne/llm-course

> How to use this notebook:
> 1. Run each level in order.
> 2. Complete the coding lab.
> 3. Run the quiz.
> 4. Check your level score.
> 5. Write notes in the Notes Vault section.
> 6. Repeat until every level is green.

In [ ]:
#@title Setup: imports, scoring, helper functions
import math, random, json, re, textwrap
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Callable, Tuple

try:
    import numpy as np
    import pandas as pd
except Exception:
    !pip -q install numpy pandas
    import numpy as np
    import pandas as pd

try:
    import sklearn
except Exception:
    !pip -q install scikit-learn
    import sklearn

progress = {}

def record_score(level: str, quiz_score: float = 0, lab_score: float = 0, notes: str = ""):
    total = round((quiz_score * 0.4) + (lab_score * 0.6), 2)
    progress[level] = {
        "quiz_score": quiz_score,
        "lab_score": lab_score,
        "total_score": total,
        "status": "PASS ✅" if total >= 70 else "RETRY 🔁",
        "notes": notes
    }
    return progress[level]

def show_progress():
    if not progress:
        print("No scores recorded yet.")
        return
    display(pd.DataFrame(progress).T)

def run_mcq(questions: List[Dict[str, Any]]):
    print("Answer format: type A/B/C/D when prompted.")
    correct = 0
    answers = {}
    for i, q in enumerate(questions, 1):
        print(f"\nQ{i}. {q['question']}")
        for k, v in q["options"].items():
            print(f"  {k}. {v}")
        ans = input("Your answer: ").strip().upper()
        answers[i] = ans
        if ans == q["answer"]:
            correct += 1
            print("✅ Correct")
        else:
            print(f"❌ Correct answer: {q['answer']} — {q.get('explanation', '')}")
    score = round(100 * correct / len(questions), 2)
    print(f"\nScore: {score}%")
    return score, answers

def assert_lab(condition: bool, points: int, message: str):
    if condition:
        print(f"✅ +{points}: {message}")
        return points
    else:
        print(f"❌ +0: {message}")
        return 0

def lab_total(points: List[int], max_points: int):
    score = round(100 * sum(points) / max_points, 2)
    print(f"Lab score: {score}%")
    return score

## Roadmap Overview

| Level | Theme | Output | Pass Criteria |
|---|---|---|---|
| 0 | AI Engineer mindset | Know what AI Engineers build | Quiz ≥ 70 |
| 1 | Python + data basics | Clean data and compute metrics | Lab + quiz ≥ 70 |
| 2 | ML fundamentals | Train/evaluate a classifier | Lab + quiz ≥ 70 |
| 3 | LLM APIs + prompt engineering | Build prompt patterns and evaluators | Lab + quiz ≥ 70 |
| 4 | Embeddings + semantic search | Build mini vector search | Lab + quiz ≥ 70 |
| 5 | RAG | Build retrieval + answer generation skeleton | Lab + quiz ≥ 70 |
| 6 | Agents + tools | Build a tool-calling mini-agent | Lab + quiz ≥ 70 |
| 7 | Evaluation + safety | Create AI eval rubric | Lab + quiz ≥ 70 |
| 8 | Deployment + LLMOps | Design production AI system | Lab + quiz ≥ 70 |
| 9 | Capstone | Build portfolio project | Rubric ≥ 70 |

# Level 0 — AI Engineer Mindset

An AI Engineer uses existing models, APIs, open-source models, retrieval, tools, evaluation, and product engineering to solve real user problems.

Your focus:
- Build useful AI applications.
- Evaluate quality.
- Handle failure cases.
- Ship safely.
- Monitor cost, latency, and quality.

In [ ]:
level0_quiz = [
    {
        "question": "What is the most practical focus of an AI Engineer?",
        "options": {"A": "Inventing new foundation model architectures", "B": "Using AI models/tools to build useful product features", "C": "Only writing SQL dashboards", "D": "Only academic research"},
        "answer": "B",
        "explanation": "AI Engineers usually apply models and tools to real products."
    },
    {
        "question": "Which skill is most important for production AI apps?",
        "options": {"A": "Prompting only", "B": "Evaluation, observability, reliability, and UX", "C": "Memorizing equations", "D": "Avoiding software engineering"},
        "answer": "B"
    },
    {
        "question": "What should you build while learning?",
        "options": {"A": "Only read theory", "B": "Small end-to-end projects", "C": "Nothing until expert", "D": "Only watch videos"},
        "answer": "B"
    }
]
# score, answers = run_mcq(level0_quiz)
# record_score("Level 0 - Mindset", quiz_score=score, lab_score=100)

# Level 1 — Python + Data Basics

## Learn
- Python functions, lists, dictionaries
- NumPy arrays
- Pandas DataFrames
- Basic data cleaning
- Simple metrics

## Code Lab
You will clean a small dataset and compute conversion metrics.

In [ ]:
# Level 1 Lab: Data cleaning and metrics

raw_events = [
    {"user_id": "u1", "event": "app_opened", "version": "1.0.0", "latency_ms": 120},
    {"user_id": "u1", "event": "ai_answered", "version": "1.0.0", "latency_ms": 950},
    {"user_id": "u2", "event": "app_opened", "version": "1.0.0", "latency_ms": None},
    {"user_id": "u2", "event": "ai_failed", "version": "1.0.0", "latency_ms": 2100},
    {"user_id": "u3", "event": "app_opened", "version": "1.1.0", "latency_ms": 100},
    {"user_id": "u3", "event": "ai_answered", "version": "1.1.0", "latency_ms": 700},
]

df = pd.DataFrame(raw_events)

# TODO 1: Fill missing latency with median latency.
df["latency_ms_clean"] = df["latency_ms"].fillna(df["latency_ms"].median())

# TODO 2: Compute success rate = ai_answered / (ai_answered + ai_failed)
answered = (df["event"] == "ai_answered").sum()
failed = (df["event"] == "ai_failed").sum()
success_rate = answered / (answered + failed)

# TODO 3: Compute p95 latency.
p95_latency = np.percentile(df["latency_ms_clean"], 95)

print(df)
print("success_rate:", success_rate)
print("p95_latency:", p95_latency)

points = []
points.append(assert_lab(df["latency_ms_clean"].isna().sum() == 0, 30, "No missing latency values"))
points.append(assert_lab(abs(success_rate - (2/3)) < 1e-9, 40, "Correct success rate"))
points.append(assert_lab(p95_latency > 0, 30, "Computed p95 latency"))
level1_lab_score = lab_total(points, 100)

In [ ]:
level1_quiz = [
    {"question": "Which Pandas object is best for table-like data?", "options": {"A": "DataFrame", "B": "Tuple", "C": "String", "D": "Set"}, "answer": "A"},
    {"question": "Why fill missing values before computing metrics?", "options": {"A": "To hide bugs", "B": "To avoid broken or biased calculations", "C": "To make data smaller", "D": "To remove columns"}, "answer": "B"},
    {"question": "What does p95 latency mean?", "options": {"A": "Average latency", "B": "95% of requests are at or below this latency", "C": "Fastest request", "D": "Total latency"}, "answer": "B"},
]
# score, answers = run_mcq(level1_quiz)
# record_score("Level 1 - Python Data", quiz_score=score, lab_score=level1_lab_score)

# Level 2 — Machine Learning Fundamentals

## Learn
- Train/test split
- Classification
- Accuracy, precision, recall
- Overfitting
- Baseline models

## Code Lab
Train a simple model to classify support tickets.

In [ ]:
# Level 2 Lab: Simple ML classifier

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

texts = [
    "app crashes after login", "payment failed at checkout", "order delayed again",
    "cannot reset password", "refund not received", "app freezes on home screen",
    "card payment declined", "delivery partner late", "login otp not received",
    "wallet refund missing", "application crashes on launch", "checkout payment error"
]
labels = [
    "tech", "payment", "delivery",
    "tech", "payment", "tech",
    "payment", "delivery", "tech",
    "payment", "tech", "payment"
]

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.25, random_state=42, stratify=labels)

clf = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)
preds = clf.predict(X_test)
acc = accuracy_score(y_test, preds)

print("Predictions:", list(zip(X_test, y_test, preds)))
print("Accuracy:", acc)
print(classification_report(y_test, preds, zero_division=0))

points = []
points.append(assert_lab(hasattr(clf, "predict"), 30, "Pipeline can predict"))
points.append(assert_lab(0 <= acc <= 1, 30, "Accuracy is valid"))
points.append(assert_lab(len(preds) == len(y_test), 40, "Predictions match test size"))
level2_lab_score = lab_total(points, 100)

In [ ]:
level2_quiz = [
    {"question": "Why use a test set?", "options": {"A": "To evaluate generalization", "B": "To train faster", "C": "To delete data", "D": "To avoid metrics"}, "answer": "A"},
    {"question": "What is overfitting?", "options": {"A": "Model performs well only on training data", "B": "Model has no data", "C": "Model is too cheap", "D": "Model is deployed"}, "answer": "A"},
    {"question": "For imbalanced classes, accuracy alone can be misleading. Which metric helps?", "options": {"A": "Precision/Recall", "B": "File size", "C": "CPU name", "D": "Line count"}, "answer": "A"},
]
# score, answers = run_mcq(level2_quiz)
# record_score("Level 2 - ML Fundamentals", quiz_score=score, lab_score=level2_lab_score)

# Level 3 — LLM APIs + Prompt Engineering

## Learn
- System/user/developer messages
- Few-shot prompting
- Structured JSON outputs
- Guardrails
- Prompt evaluation

## Code Lab
Create a deterministic prompt template and validate model-like output.

> This lab avoids paid API calls. Later, replace `mock_llm()` with OpenAI, Anthropic, Gemini, or local Ollama calls.

In [ ]:
# Level 3 Lab: Prompt template + output validation

def build_support_prompt(ticket: str) -> str:
    return f'''
You are a support triage AI.
Classify the ticket into one of: tech, payment, delivery, account.
Return strict JSON with keys: category, priority, reason.

Ticket: {ticket}
'''.strip()

def mock_llm(prompt: str) -> str:
    p = prompt.lower()
    if "payment" in p or "card" in p or "refund" in p:
        return json.dumps({"category": "payment", "priority": "high", "reason": "Payment-related issue"})
    if "crash" in p or "freeze" in p:
        return json.dumps({"category": "tech", "priority": "high", "reason": "App stability issue"})
    return json.dumps({"category": "account", "priority": "medium", "reason": "General account issue"})

def validate_output(s: str) -> Tuple[bool, Dict[str, Any]]:
    try:
        obj = json.loads(s)
        required = {"category", "priority", "reason"}
        valid_categories = {"tech", "payment", "delivery", "account"}
        ok = required.issubset(obj.keys()) and obj["category"] in valid_categories
        return ok, obj
    except Exception:
        return False, {}

prompt = build_support_prompt("My card payment failed and refund is missing")
response = mock_llm(prompt)
ok, parsed = validate_output(response)

print(prompt)
print(response)
print(ok, parsed)

points = []
points.append(assert_lab("Return strict JSON" in prompt, 25, "Prompt requests strict JSON"))
points.append(assert_lab(ok, 45, "Output validates"))
points.append(assert_lab(parsed.get("category") == "payment", 30, "Correct category"))
level3_lab_score = lab_total(points, 100)

In [ ]:
level3_quiz = [
    {"question": "Why ask for JSON output?", "options": {"A": "Easier parsing and automation", "B": "Looks fancy", "C": "Always cheaper", "D": "Avoids evaluation"}, "answer": "A"},
    {"question": "What is few-shot prompting?", "options": {"A": "No examples", "B": "Providing examples in the prompt", "C": "Training a model", "D": "Deleting context"}, "answer": "B"},
    {"question": "What should you do with LLM outputs before using in code?", "options": {"A": "Trust blindly", "B": "Validate and handle failures", "C": "Print only", "D": "Ignore schema"}, "answer": "B"},
]
# score, answers = run_mcq(level3_quiz)
# record_score("Level 3 - LLM Prompting", quiz_score=score, lab_score=level3_lab_score)

# Level 4 — Embeddings + Semantic Search

## Learn
- Embeddings convert text to vectors
- Similarity search retrieves relevant chunks
- Vector databases store and search embeddings
- Useful for search, recommendations, RAG

## Code Lab
Build a tiny semantic search engine using TF-IDF as a lightweight embedding substitute.

In [ ]:
# Level 4 Lab: Mini semantic search

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "RAG uses retrieval to provide context to a language model.",
    "Prompt engineering improves model instructions and output quality.",
    "Vector databases store embeddings for semantic search.",
    "LLMOps includes monitoring, cost tracking, evaluation, and deployment.",
    "Agents use tools to take actions and complete multi-step tasks."
]

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(docs)

def search(query: str, top_k: int = 2):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_vectors)[0]
    ranked = np.argsort(sims)[::-1][:top_k]
    return [(int(i), docs[i], float(sims[i])) for i in ranked]

results = search("how do I retrieve context for an LLM?", top_k=2)
results

In [ ]:
points = []
points.append(assert_lab(len(results) == 2, 25, "Returns top_k results"))
points.append(assert_lab("RAG" in results[0][1] or "retrieval" in results[0][1].lower(), 50, "Top result is retrieval-related"))
points.append(assert_lab(all(isinstance(r[2], float) for r in results), 25, "Each result has similarity score"))
level4_lab_score = lab_total(points, 100)

In [ ]:
level4_quiz = [
    {"question": "What is an embedding?", "options": {"A": "Text represented as a vector", "B": "A database password", "C": "A UI button", "D": "A Python loop"}, "answer": "A"},
    {"question": "What does cosine similarity compare?", "options": {"A": "Vector direction similarity", "B": "File age", "C": "Model cost only", "D": "Token count only"}, "answer": "A"},
    {"question": "Why use vector search in RAG?", "options": {"A": "To retrieve relevant context", "B": "To delete documents", "C": "To avoid prompts", "D": "To train GPUs"}, "answer": "A"},
]
# score, answers = run_mcq(level4_quiz)
# record_score("Level 4 - Embeddings Search", quiz_score=score, lab_score=level4_lab_score)

# Level 5 — RAG: Retrieval-Augmented Generation

## Learn
- Chunking
- Retrieval
- Context injection
- Citation-aware answers
- Hallucination reduction
- Evaluation: faithfulness, relevance, answer correctness

## Code Lab
Build a toy RAG pipeline.

In [ ]:
# Level 5 Lab: Toy RAG pipeline

knowledge_base = [
    {"id": "doc1", "text": "Abhyanga is an Ayurvedic oil massage practice often used for relaxation and wellbeing."},
    {"id": "doc2", "text": "RAG means Retrieval-Augmented Generation. It retrieves documents before generating an answer."},
    {"id": "doc3", "text": "AI agents can call tools such as search, calculators, databases, and APIs."},
    {"id": "doc4", "text": "LLMOps tracks latency, cost, quality, safety, and failures in production AI applications."},
]

kb_texts = [d["text"] for d in knowledge_base]
rag_vectorizer = TfidfVectorizer()
kb_vectors = rag_vectorizer.fit_transform(kb_texts)

def retrieve(query: str, top_k: int = 2):
    q_vec = rag_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, kb_vectors)[0]
    ranked = np.argsort(sims)[::-1][:top_k]
    return [knowledge_base[i] | {"score": float(sims[i])} for i in ranked]

def generate_answer(query: str, contexts: List[Dict[str, Any]]) -> str:
    context_text = "\n".join([f"[{c['id']}] {c['text']}" for c in contexts])
    return f"Answer based on retrieved context:\n{context_text}\n\nQuestion: {query}\nDraft answer: RAG retrieves relevant documents and uses them as context before generating an answer."

query = "What is RAG?"
contexts = retrieve(query)
answer = generate_answer(query, contexts)

print(contexts)
print(answer)

points = []
points.append(assert_lab(len(contexts) == 2, 20, "Retrieves contexts"))
points.append(assert_lab(any("RAG" in c["text"] for c in contexts), 40, "Retrieves RAG document"))
points.append(assert_lab("[doc" in answer, 20, "Answer includes source IDs"))
points.append(assert_lab("retrieves" in answer.lower(), 20, "Answer uses retrieved context"))
level5_lab_score = lab_total(points, 100)

In [ ]:
level5_quiz = [
    {"question": "What is the main purpose of RAG?", "options": {"A": "Add relevant external context before answering", "B": "Make prompts shorter only", "C": "Avoid retrieval", "D": "Replace databases completely"}, "answer": "A"},
    {"question": "What is a common RAG failure?", "options": {"A": "Retrieving irrelevant chunks", "B": "Having documents", "C": "Using citations", "D": "Chunking text"}, "answer": "A"},
    {"question": "Which metric helps evaluate retrieved chunks?", "options": {"A": "Context relevance", "B": "Laptop color", "C": "Screen size", "D": "Code font"}, "answer": "A"},
]
# score, answers = run_mcq(level5_quiz)
# record_score("Level 5 - RAG", quiz_score=score, lab_score=level5_lab_score)

# Level 6 — Agents + Tool Calling

## Learn
- An agent chooses actions to complete a task
- Tools can be calculators, search, APIs, databases, Slack, GitHub, Firebase, BigQuery, etc.
- Good agents need bounded goals, tool permissions, memory, evaluation, and observability

## Code Lab
Build a tiny rule-based tool-calling agent.

In [ ]:
# Level 6 Lab: Mini tool-calling agent

def calculator_tool(expression: str) -> str:
    allowed = re.fullmatch(r"[0-9+\-*/(). ]+", expression)
    if not allowed:
        return "Invalid expression"
    return str(eval(expression, {"__builtins__": {}}))

def search_docs_tool(query: str) -> str:
    results = retrieve(query, top_k=1)
    return results[0]["text"] if results else "No result"

tools = {
    "calculator": calculator_tool,
    "search_docs": search_docs_tool
}

def mini_agent(task: str) -> Dict[str, Any]:
    lower = task.lower()
    if "calculate" in lower or re.search(r"\d+\s*[+\-*/]\s*\d+", lower):
        expr = re.findall(r"[0-9+\-*/(). ]+", task)[0].strip()
        return {"tool": "calculator", "input": expr, "output": tools["calculator"](expr)}
    else:
        return {"tool": "search_docs", "input": task, "output": tools["search_docs"](task)}

agent_result_1 = mini_agent("calculate 12 * 8 + 4")
agent_result_2 = mini_agent("what does LLMOps track?")

print(agent_result_1)
print(agent_result_2)

points = []
points.append(assert_lab(agent_result_1["tool"] == "calculator", 30, "Agent selected calculator"))
points.append(assert_lab(agent_result_1["output"] == "100", 30, "Calculator result correct"))
points.append(assert_lab(agent_result_2["tool"] == "search_docs", 20, "Agent selected search_docs"))
points.append(assert_lab("LLMOps" in agent_result_2["output"], 20, "Search tool returned relevant info"))
level6_lab_score = lab_total(points, 100)

In [ ]:
level6_quiz = [
    {"question": "What makes an AI workflow agentic?", "options": {"A": "It can choose actions/tools toward a goal", "B": "It has a logo", "C": "It always uses a GPU", "D": "It only summarizes text"}, "answer": "A"},
    {"question": "Why restrict tool permissions?", "options": {"A": "For safety and control", "B": "To make it slower", "C": "To avoid logs", "D": "To remove tests"}, "answer": "A"},
    {"question": "Which is a good agent demo for a mobile engineer?", "options": {"A": "Crash-rate investigator that queries metrics and posts Slack summary", "B": "Random chatbot only", "C": "Static HTML page", "D": "Manual spreadsheet copy"}, "answer": "A"},
]
# score, answers = run_mcq(level6_quiz)
# record_score("Level 6 - Agents Tools", quiz_score=score, lab_score=level6_lab_score)

# Level 7 — Evaluation + Safety

## Learn
- Unit tests for prompts
- Golden datasets
- Human review
- LLM-as-judge with caution
- Regression tests
- Hallucination, bias, privacy, prompt injection, unsafe actions

## Code Lab
Create a simple evaluator for AI answers.

In [ ]:
# Level 7 Lab: Evaluator

eval_cases = [
    {
        "question": "What is RAG?",
        "expected_keywords": ["retrieval", "context", "generate"],
        "answer": "RAG retrieves relevant context before generating an answer."
    },
    {
        "question": "What does LLMOps track?",
        "expected_keywords": ["latency", "cost", "quality"],
        "answer": "LLMOps tracks latency, cost, quality, safety, and failures."
    }
]

def keyword_eval(case):
    answer = case["answer"].lower()
    hits = sum(1 for kw in case["expected_keywords"] if kw.lower() in answer)
    return {
        "question": case["question"],
        "hits": hits,
        "total": len(case["expected_keywords"]),
        "score": round(100 * hits / len(case["expected_keywords"]), 2)
    }

eval_results = [keyword_eval(c) for c in eval_cases]
display(pd.DataFrame(eval_results))

avg_eval_score = np.mean([r["score"] for r in eval_results])

points = []
points.append(assert_lab(avg_eval_score >= 70, 50, "Average eval score >= 70"))
points.append(assert_lab(all("score" in r for r in eval_results), 25, "Each case has score"))
points.append(assert_lab(len(eval_results) >= 2, 25, "At least two eval cases"))
level7_lab_score = lab_total(points, 100)

In [ ]:
level7_quiz = [
    {"question": "Why create golden datasets?", "options": {"A": "To regression-test AI quality", "B": "To decorate notebooks", "C": "To reduce RAM only", "D": "To avoid users"}, "answer": "A"},
    {"question": "What is prompt injection?", "options": {"A": "Input trying to override system/developer instructions", "B": "A database index", "C": "A GPU error", "D": "A CSS issue"}, "answer": "A"},
    {"question": "Which production metric matters for AI apps?", "options": {"A": "Latency, cost, quality, safety", "B": "Only font size", "C": "Only code lines", "D": "Only page views"}, "answer": "A"},
]
# score, answers = run_mcq(level7_quiz)
# record_score("Level 7 - Eval Safety", quiz_score=score, lab_score=level7_lab_score)

# Level 8 — Deployment + LLMOps

## Learn
- API design
- Caching
- Rate limits
- Secrets management
- Observability
- Cost controls
- Prompt/model versioning
- Rollbacks
- A/B testing
- Feedback loops

## Architecture Exercise
Design a production AI feature.

In [ ]:
# Level 8 Lab: Production readiness checklist

production_checklist = {
    "api_contract": True,
    "input_validation": True,
    "output_schema_validation": True,
    "eval_dataset": True,
    "logging": True,
    "latency_monitoring": True,
    "cost_monitoring": True,
    "fallback_model_or_message": True,
    "human_review_path": True,
    "privacy_review": True,
}

def readiness_score(checklist: Dict[str, bool]) -> float:
    return round(100 * sum(checklist.values()) / len(checklist), 2)

level8_lab_score = readiness_score(production_checklist)
print("Production readiness score:", level8_lab_score)
display(pd.DataFrame([production_checklist]).T.rename(columns={0: "ready"}))

In [ ]:
level8_quiz = [
    {"question": "Why version prompts?", "options": {"A": "To track quality changes and rollback safely", "B": "To make text longer", "C": "To avoid deployment", "D": "To remove tests"}, "answer": "A"},
    {"question": "What should happen if the AI service fails?", "options": {"A": "Fallback path or graceful error", "B": "Crash the whole app", "C": "Ignore user", "D": "Delete logs"}, "answer": "A"},
    {"question": "What should you monitor?", "options": {"A": "Cost, latency, quality, failures, safety", "B": "Only CPU temperature", "C": "Only daily active users", "D": "Nothing"}, "answer": "A"},
]
# score, answers = run_mcq(level8_quiz)
# record_score("Level 8 - Deployment LLMOps", quiz_score=score, lab_score=level8_lab_score)

# Level 9 — Capstone Projects

Pick **one** project and build it end-to-end.

## Best capstone ideas for you

### 1. Crash Rate Investigation Agent
- Input: app version
- Tools: Firebase / BigQuery / Grafana data source
- Actions: compare latest version vs previous versions
- Output: Slack alert with risk summary and evidence
- Portfolio value: very high for your mobile/CI background

### 2. AI News Digest PWA
- Input: sources + topics
- Tools: RSS/search APIs + LLM summarizer
- Output: daily digest, categories, importance score
- Portfolio value: strong product demo

### 3. PR Review Assistant for Flutter
- Input: GitHub PR
- Tools: GitHub API, static analysis, test results
- Output: code review comments, risk score, size impact
- Portfolio value: very high for DevX/CI profile

### 4. RAG Knowledge Assistant
- Input: docs, PDFs, tickets, incidents
- Tools: embeddings, vector DB, citations
- Output: grounded answers with source references
- Portfolio value: classic AI engineer project

## Capstone Rubric

| Area | Points |
|---|---:|
| Clear problem + users | 10 |
| Working app/API | 20 |
| Retrieval/tool use/LLM integration | 20 |
| Evaluation dataset + metrics | 20 |
| Safety/fallbacks | 10 |
| Observability/cost tracking | 10 |
| Demo/readme/video | 10 |

In [ ]:
# Capstone self-evaluation

capstone = {
    "clear_problem_users": 0,      # 0-10
    "working_app_or_api": 0,       # 0-20
    "llm_retrieval_or_tools": 0,   # 0-20
    "evaluation_metrics": 0,       # 0-20
    "safety_fallbacks": 0,         # 0-10
    "observability_cost": 0,       # 0-10
    "demo_readme_video": 0,        # 0-10
}

def capstone_score(rubric):
    score = sum(rubric.values())
    status = "PORTFOLIO READY ✅" if score >= 70 else "IMPROVE 🔁"
    print(f"Capstone score: {score}/100 — {status}")
    return score

# Fill the rubric above, then run:
# capstone_score(capstone)

# Existing GitHub Projects Worth Using

| Repo | Best For | How to Use |
|---|---|---|
| `PavanMudigonda/zero-to-ai` | Huge notebook library: Python, DS, math, deep learning, LLMs, RAG, agents, MLOps | Use as practice notebook source |
| `dswh/ai-engineer-roadmap` | Simple 3-stage AI engineer roadmap: beginner → intermediate → advanced | Use as quick high-level checklist |
| `AgenticAiLabs/Ai-Engineering-Roadmap` | OSSU-style open curriculum with programming, math, ML, DL, LLMs, agents, RAG | Use as long-form curriculum |
| `alexeygrigorev/ai-engineering-field-guide` | Job/interview/portfolio reality based on job descriptions | Use for interview prep and portfolio direction |
| `mlabonne/llm-course` | LLM fundamentals, tools, fine-tuning, alignment-style topics | Use for deeper LLM specialization |

# Notes Vault

Use this section as your permanent AI Engineer notebook.

## Concept Note Template

### Concept
Write the concept name.

### Explain like I am 10
Write a very simple explanation.

### Technical definition
Write a precise definition.

### Code example
Paste minimal code.

### Real-world use case
Where would you use this in a product?

### Common mistakes
List pitfalls.

### Evaluation checklist
How do you know it works?

### Links
Add references.

---

## Notes Index

| Topic | Status | Confidence 1-5 | Last Reviewed |
|---|---|---:|---|
| Python for AI | Not started | 1 | |
| Pandas/NumPy | Not started | 1 | |
| ML basics | Not started | 1 | |
| LLM APIs | Not started | 1 | |
| Prompt engineering | Not started | 1 | |
| Embeddings | Not started | 1 | |
| Vector DB | Not started | 1 | |
| RAG | Not started | 1 | |
| Agents | Not started | 1 | |
| LangGraph | Not started | 1 | |
| Evaluation | Not started | 1 | |
| LLMOps | Not started | 1 | |
| Deployment | Not started | 1 | |

In [ ]:
# Final dashboard
show_progress()

# Suggested manual schedule:
schedule = pd.DataFrame([
    {"Week": "1", "Focus": "Level 0-1: Mindset + Python/Data", "Output": "Metrics notebook"},
    {"Week": "2", "Focus": "Level 2: ML basics", "Output": "Classifier"},
    {"Week": "3", "Focus": "Level 3: LLM prompting", "Output": "Prompt evaluator"},
    {"Week": "4", "Focus": "Level 4-5: Embeddings + RAG", "Output": "Mini RAG app"},
    {"Week": "5", "Focus": "Level 6: Agents/tools", "Output": "Tool-calling agent"},
    {"Week": "6", "Focus": "Level 7-8: Eval + LLMOps", "Output": "Production checklist + eval suite"},
    {"Week": "7-8", "Focus": "Level 9: Capstone", "Output": "Portfolio project"},
])
display(schedule)